In [ ]:
#cell-1

import os, sys, time, json, csv, random, subprocess, warnings, traceback
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as tvm

warnings.filterwarnings("ignore")

# ---------------- GLOBAL CONFIG ----------------
GPU_INDEX = 0
DEVICE = torch.device(f"cuda:{GPU_INDEX}")

MODELS = [
    "mobilenet_v3_small",
    "efficientnet_b0",
    "resnet18",
    "resnet50",
    "convnext_tiny",
    "vit_b_16",
]
BATCH_SIZES = [1, 2, 4, 8, 16, 32, 64, 128]
PRECISIONS = ["fp32", "fp16"]

MEASUREMENT_SECONDS = 60.0
WARMUP_ITERS = 30
POWER_SAMPLE_INTERVAL_S = 0.1
IDLE_BASELINE_SECONDS = 2.0
IDLE_STABILITY_THRESHOLD_W = 5.0
MIN_POWER_SAMPLES_REQUIRED = 30

RESULTS_DIR = Path("/kaggle/working/results")
TRACES_DIR = Path("/kaggle/working/traces")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TRACES_DIR.mkdir(parents=True, exist_ok=True)
SAVE_TRACES = True
RESULTS_CSV = RESULTS_DIR / "all_sessions_results.csv"

# ---- COMMIT-STAGE TOGGLES ----
# All OFF — this run only loads the final merged data and produces plots/analysis.
RUN_ENERGY_SWEEP = False
RUN_ACCURACY_EVAL = False
RUN_EXTRA_SESSION = False


RUN_CODECARBON_VALIDATION = True     # set True only when you want to (re)run the CodeCarbon comparison cells


# ---- PULL IN THE TRUE FINAL MERGED DATASET ----
FINAL_DATASET_DIR = "/kaggle/input/datasets/sumiyaahasan/final-5merged"

import shutil

# Energy data (5 sessions, 480 rows)
shutil.copy(f"{FINAL_DATASET_DIR}/all_sessions_results.csv", RESULTS_CSV)
print("Copied final energy results (5 sessions) into", RESULTS_CSV)

# Accuracy data — copy AND rename to accuracy_all_models.csv so existing Cell 13/16
# code (which looks for that exact filename) picks it up without modification.
shutil.copy(f"{FINAL_DATASET_DIR}/accuracy_imagenette_imagewoof_combined.csv",
            RESULTS_DIR / "accuracy_all_models.csv")
print("Copied final accuracy results (Imagenette+Imagewoof) as accuracy_all_models.csv")

# Precomputed baseline comparison and McNemar's test — already correct, just carry them forward
shutil.copy(f"{FINAL_DATASET_DIR}/baseline_comparison.csv", RESULTS_DIR / "baseline_comparison.csv")
shutil.copy(f"{FINAL_DATASET_DIR}/mcnemar_test.csv", RESULTS_DIR / "mcnemar_test.csv")
print("Copied precomputed baseline_comparison.csv and mcnemar_test.csv")

RESULT_FIELDS = [
    "session_id", "random_seed", "timestamp",
    "model", "batch_size", "precision",
    "measurement_seconds", "num_batches", "total_samples",
    "mean_latency_s", "median_latency_s", "throughput_samples_s",
    "energy_j", "energy_per_sample_j",
    "average_power_w", "min_power_w", "max_power_w",
    "peak_memory_gb",
    "average_temperature_c", "average_sm_clock_mhz", "average_utilization_percent",
    "idle_before_w", "idle_after_w", "idle_difference_w",
    "throttle_active_fraction", "power_cap_fraction",
    "thermal_slowdown_fraction", "hardware_slowdown_fraction",
    "num_power_samples", "valid", "warnings",
]

print("Config loaded. Device target:", DEVICE)

In [ ]:
#cell-2
def run_nvidia_smi(query_fields, index=GPU_INDEX):
    """Run a single nvidia-smi query and return the raw stdout line for our GPU index."""
    cmd = [
        "nvidia-smi",
        f"--query-gpu={query_fields}",
        "--format=csv,noheader,nounits",
        f"-i", str(index),
    ]
    out = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
    return out.stdout.strip()

assert torch.cuda.is_available(), "CUDA not available"
assert torch.cuda.device_count() >= 1, "No CUDA devices found"
torch.cuda.set_device(GPU_INDEX)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Using device index:", GPU_INDEX, "->", torch.cuda.get_device_name(GPU_INDEX))

env_check = run_nvidia_smi(
    "index,name,power.limit,enforced.power.limit,power.draw,temperature.gpu,"
    "clocks.current.sm,utilization.gpu,clocks_throttle_reasons.active"
)
print("nvidia-smi snapshot (GPU {}):".format(GPU_INDEX))
print(env_check)

# Confirm energy.draw is unavailable (documented limitation, not re-tested destructively)
try:
    e = run_nvidia_smi("energy.draw")
    print("energy.draw raw output:", repr(e), "-> NOT used as primary metric regardless.")
except Exception as ex:
    print("energy.draw query failed as expected:", ex)

print("\nEnvironment verification complete. Proceeding with power.draw integration methodology.")

In [ ]:
#Cell 3 — Model definitions/loading
def build_model(name: str) -> nn.Module:
    """Return an eval-mode torchvision model with random/pretrained-agnostic weights.
    Weights='DEFAULT' is used for accuracy runs; energy/latency runs don't depend on weight values."""
    ctor_map = {
        "mobilenet_v3_small": tvm.mobilenet_v3_small,
        "efficientnet_b0": tvm.efficientnet_b0,
        "resnet18": tvm.resnet18,
        "resnet50": tvm.resnet50,
        "convnext_tiny": tvm.convnext_tiny,
        "vit_b_16": tvm.vit_b_16,
    }
    if name not in ctor_map:
        raise ValueError(f"Unknown model name: {name}")
    model = ctor_map[name](weights="DEFAULT")
    model.eval()
    return model

def load_model_to_device(name: str, precision: str) -> nn.Module:
    model = build_model(name)
    model = model.to(DEVICE)
    if precision == "fp16":
        model = model.half()
    elif precision != "fp32":
        raise ValueError(f"Unknown precision: {precision}")
    for p in model.parameters():
        p.requires_grad_(False)
    return model

def get_preprocessing_transform(name: str):
    """Each torchvision model's default transform, for the accuracy pipeline only."""
    weights_map = {
        "mobilenet_v3_small": tvm.MobileNet_V3_Small_Weights.DEFAULT,
        "efficientnet_b0": tvm.EfficientNet_B0_Weights.DEFAULT,
        "resnet18": tvm.ResNet18_Weights.DEFAULT,
        "resnet50": tvm.ResNet50_Weights.DEFAULT,
        "convnext_tiny": tvm.ConvNeXt_Tiny_Weights.DEFAULT,
        "vit_b_16": tvm.ViT_B_16_Weights.DEFAULT,
    }
    return weights_map[name].transforms()

# quick sanity load of one model (cheap, not a full test suite)
_test_model = load_model_to_device("resnet18", "fp32")
print("Sample model loaded OK:", type(_test_model).__name__)
del _test_model
torch.cuda.empty_cache()

In [ ]:
#Cell 4 — Configuration grid + randomization

def build_configuration_grid(models=MODELS, batch_sizes=BATCH_SIZES, precisions=PRECISIONS):
    grid = []
    for m in models:
        for b in batch_sizes:
            for p in precisions:
                grid.append({"model": m, "batch_size": b, "precision": p})
    return grid

def randomize_grid(grid, seed):
    rng = random.Random(seed)
    shuffled = grid.copy()
    rng.shuffle(shuffled)
    return shuffled

def new_session(session_id: str = None, seed: int = None):
    if session_id is None:
        session_id = "session_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    if seed is None:
        seed = random.randint(0, 2**31 - 1)
    grid = build_configuration_grid()
    grid = randomize_grid(grid, seed)
    print(f"Session '{session_id}' created. Seed={seed}. {len(grid)} configurations queued.")
    return session_id, seed, grid

if RUN_ENERGY_SWEEP:
    SESSION_ID, SEED, CONFIG_GRID = new_session()
    print("SESSION_ID:", SESSION_ID, "| SEED:", SEED, "-- SAVE THESE, you need them to resume.")
else:
    SESSION_ID, SEED = None, None
    print("RUN_ENERGY_SWEEP is False — skipping new session creation for this run.")

In [ ]:
#Cell 5 — Monitoring utilities (nvidia-smi sampler)

import threading

NVSMI_QUERY_FIELDS = (
    "power.draw,temperature.gpu,clocks.current.sm,utilization.gpu,"
    "clocks_throttle_reasons.active,clocks_throttle_reasons.sw_power_cap,"
    "clocks_throttle_reasons.hw_slowdown,clocks_throttle_reasons.hw_thermal_slowdown,"
    "clocks_throttle_reasons.hw_power_brake_slowdown,"
    "clocks_throttle_reasons.gpu_idle,clocks_throttle_reasons.applications_clocks_setting,"
    "clocks_throttle_reasons.sync_boost"
)

def _parse_bool_field(raw):
    raw = raw.strip()
    if raw == "Active":
        return True
    if raw == "Not Active":
        return False
    return "unavailable"

def sample_gpu_state_once(index=GPU_INDEX):
    """One nvidia-smi query -> dict. Never raises; missing fields become 'unavailable'."""
    try:
        line = run_nvidia_smi(NVSMI_QUERY_FIELDS, index=index)
        parts = [x.strip() for x in line.split(",")]
        if len(parts) < 12:
            raise ValueError("Unexpected nvidia-smi output: " + line)
        power_w = float(parts[0]) if parts[0] not in ("", "[N/A]") else float("nan")
        temp_c = float(parts[1]) if parts[1] not in ("", "[N/A]") else float("nan")
        sm_clock = float(parts[2]) if parts[2] not in ("", "[N/A]") else float("nan")
        util = float(parts[3]) if parts[3] not in ("", "[N/A]") else float("nan")
        return {
            "timestamp": time.time(),
            "power_w": power_w,
            "temperature_c": temp_c,
            "sm_clock_mhz": sm_clock,
            "gpu_utilization_percent": util,
            "throttle_active_raw": parts[4],
            "sw_power_cap": _parse_bool_field(parts[5]),
            "hw_slowdown": _parse_bool_field(parts[6]),
            "hw_thermal_slowdown": _parse_bool_field(parts[7]),
            "hw_power_brake_slowdown": _parse_bool_field(parts[8]),
            "gpu_idle": _parse_bool_field(parts[9]),
            "applications_clocks_setting": _parse_bool_field(parts[10]),
            "sync_boost": _parse_bool_field(parts[11]),
        }
    except Exception as ex:
        return {
            "timestamp": time.time(), "power_w": float("nan"), "temperature_c": float("nan"),
            "sm_clock_mhz": float("nan"), "gpu_utilization_percent": float("nan"),
            "throttle_active_raw": "unavailable", "sw_power_cap": "unavailable",
            "hw_slowdown": "unavailable", "hw_thermal_slowdown": "unavailable",
            "hw_power_brake_slowdown": "unavailable", "gpu_idle": "unavailable",
            "applications_clocks_setting": "unavailable", "sync_boost": "unavailable",
            "_error": str(ex),
        }

class PowerSampler:
    """Background thread polling nvidia-smi at a fixed interval."""
    def __init__(self, interval_s=POWER_SAMPLE_INTERVAL_S, index=GPU_INDEX):
        self.interval_s = interval_s
        self.index = index
        self._samples = []
        self._stop_event = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop_event.is_set():
            self._samples.append(sample_gpu_state_once(self.index))
            time.sleep(self.interval_s)

    def start(self):
        self._samples = []
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=5)
        return self._samples

def idle_baseline(duration_s=IDLE_BASELINE_SECONDS):
    sampler = PowerSampler()
    sampler.start()
    time.sleep(duration_s)
    samples = sampler.stop()
    powers = [s["power_w"] for s in samples if not np.isnan(s["power_w"])]
    return float(np.mean(powers)) if powers else float("nan")

print("Monitoring utilities defined.")

In [ ]:
#Cell 6 — Energy calculation

def integrate_energy_trapezoidal(samples):
    """samples: list of dicts with 'timestamp' and 'power_w', time-ordered.
    Returns (energy_joules, num_valid_samples)."""
    clean = [s for s in samples if not np.isnan(s.get("power_w", float("nan")))]
    clean.sort(key=lambda s: s["timestamp"])
    if len(clean) < 2:
        return 0.0, len(clean)
    energy = 0.0
    for i in range(1, len(clean)):
        p_prev, p_curr = clean[i-1]["power_w"], clean[i]["power_w"]
        t_prev, t_curr = clean[i-1]["timestamp"], clean[i]["timestamp"]
        dt = t_curr - t_prev
        if dt <= 0:
            continue
        energy += ((p_prev + p_curr) / 2.0) * dt
    return energy, len(clean)

print("Energy integration function defined (trapezoidal rule, Joules).")

In [ ]:
#Cell 7 — Validity logic

def evaluate_validity(power_samples, num_power_samples, idle_before, idle_after,
                       num_batches, measurement_seconds, oom_occurred, non_finite_occurred):
    """Returns (valid: bool, warnings: list[str], fractions: dict)."""
    warnings_list = []

    def frac_true(field):
        vals = [s[field] for s in power_samples if s.get(field) in (True, False)]
        if not vals:
            return float("nan")
        return sum(1 for v in vals if v) / len(vals)

    throttle_active_frac = (
        sum(1 for s in power_samples if s.get("throttle_active_raw") not in ("0x0000000000000000", "unavailable")) 
        / len(power_samples) if power_samples else float("nan")
    )
    power_cap_frac = frac_true("sw_power_cap")
    thermal_frac = frac_true("hw_thermal_slowdown")
    hw_slowdown_frac = frac_true("hw_slowdown")

    valid = True

    if oom_occurred:
        valid = False
        warnings_list.append("OOM")
    if non_finite_occurred:
        valid = False
        warnings_list.append("non_finite_outputs")
    if num_power_samples < MIN_POWER_SAMPLES_REQUIRED:
        valid = False
        warnings_list.append("insufficient_power_samples")
    if measurement_seconds < MEASUREMENT_SECONDS * 0.95:
        valid = False
        warnings_list.append("insufficient_measurement_duration")
    if not np.isnan(idle_before) and not np.isnan(idle_after):
        if abs(idle_after - idle_before) > IDLE_STABILITY_THRESHOLD_W:
            warnings_list.append("unstable_idle_baseline")  # flagged, not auto-invalidated
    if not np.isnan(thermal_frac) and thermal_frac > 0.10:
        valid = False
        warnings_list.append("hw_thermal_slowdown_exceeded_10pct")
    if not np.isnan(hw_slowdown_frac) and hw_slowdown_frac > 0.10:
        valid = False
        warnings_list.append("hw_slowdown_exceeded_10pct")
    # generic throttle flag alone (e.g. applications_clocks_setting / gpu_idle bit) never invalidates by itself
    if not np.isnan(throttle_active_frac) and throttle_active_frac > 0:
        warnings_list.append(f"generic_throttle_flag_nonzero_fraction={throttle_active_frac:.3f}")

    fractions = {
        "throttle_active_fraction": throttle_active_frac,
        "power_cap_fraction": power_cap_frac,
        "thermal_slowdown_fraction": thermal_frac,
        "hardware_slowdown_fraction": hw_slowdown_frac,
    }
    return valid, warnings_list, fractions

print("Validity logic defined. Thresholds: MIN_POWER_SAMPLES_REQUIRED =",
      MIN_POWER_SAMPLES_REQUIRED, "| IDLE_STABILITY_THRESHOLD_W =", IDLE_STABILITY_THRESHOLD_W,
      "| thermal/hw_slowdown invalidation threshold = 10% of samples")

In [ ]:
#Cell 8 — Single-configuration measurement function

def run_single_configuration(model_name, batch_size, precision,
                              measurement_seconds=MEASUREMENT_SECONDS,
                              warmup_iters=WARMUP_ITERS,
                              save_trace=SAVE_TRACES, session_id="adhoc"):
    result = {f: None for f in RESULT_FIELDS}
    result.update({
        "model": model_name, "batch_size": batch_size, "precision": precision,
        "session_id": session_id, "timestamp": datetime.utcnow().isoformat(),
    })
    warnings_list = []
    oom_occurred = False
    non_finite_occurred = False

    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(DEVICE)

        model = load_model_to_device(model_name, precision)
        dtype = torch.float16 if precision == "fp16" else torch.float32
        x = torch.randn(batch_size, 3, 224, 224, device=DEVICE, dtype=dtype)

        # warm-up (discarded)
        with torch.no_grad():
            for _ in range(warmup_iters):
                out = model(x)
        torch.cuda.synchronize(DEVICE)

        idle_before = idle_baseline()

        sampler = PowerSampler()
        sampler.start()

        batch_latencies = []
        num_batches = 0
        start_time = time.time()
        with torch.no_grad():
            while (time.time() - start_time) < measurement_seconds:
                b0 = time.time()
                out = model(x)
                torch.cuda.synchronize(DEVICE)
                b1 = time.time()
                batch_latencies.append(b1 - b0)
                num_batches += 1
                if not torch.isfinite(out).all():
                    non_finite_occurred = True
        end_time = time.time()

        power_samples = sampler.stop()
        idle_after = idle_baseline()

        actual_measurement_seconds = end_time - start_time
        total_samples = num_batches * batch_size
        energy_j, num_power_samples = integrate_energy_trapezoidal(power_samples)
        energy_per_sample = energy_j / total_samples if total_samples > 0 else float("nan")

        powers = [s["power_w"] for s in power_samples if not np.isnan(s["power_w"])]
        temps = [s["temperature_c"] for s in power_samples if not np.isnan(s["temperature_c"])]
        clocks = [s["sm_clock_mhz"] for s in power_samples if not np.isnan(s["sm_clock_mhz"])]
        utils = [s["gpu_utilization_percent"] for s in power_samples if not np.isnan(s["gpu_utilization_percent"])]

        peak_mem_gb = torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)

        valid, extra_warnings, fractions = evaluate_validity(
            power_samples, num_power_samples, idle_before, idle_after,
            num_batches, actual_measurement_seconds, oom_occurred, non_finite_occurred
        )
        warnings_list.extend(extra_warnings)

        result.update({
            "measurement_seconds": actual_measurement_seconds,
            "num_batches": num_batches,
            "total_samples": total_samples,
            "mean_latency_s": float(np.mean(batch_latencies)) if batch_latencies else float("nan"),
            "median_latency_s": float(np.median(batch_latencies)) if batch_latencies else float("nan"),
            "throughput_samples_s": total_samples / actual_measurement_seconds if actual_measurement_seconds > 0 else float("nan"),
            "energy_j": energy_j,
            "energy_per_sample_j": energy_per_sample,
            "average_power_w": float(np.mean(powers)) if powers else float("nan"),
            "min_power_w": float(np.min(powers)) if powers else float("nan"),
            "max_power_w": float(np.max(powers)) if powers else float("nan"),
            "peak_memory_gb": peak_mem_gb,
            "average_temperature_c": float(np.mean(temps)) if temps else float("nan"),
            "average_sm_clock_mhz": float(np.mean(clocks)) if clocks else float("nan"),
            "average_utilization_percent": float(np.mean(utils)) if utils else float("nan"),
            "idle_before_w": idle_before,
            "idle_after_w": idle_after,
            "idle_difference_w": (idle_after - idle_before) if (not np.isnan(idle_before) and not np.isnan(idle_after)) else float("nan"),
            "num_power_samples": num_power_samples,
            "valid": valid,
            "warnings": ";".join(warnings_list) if warnings_list else "",
            **fractions,
        })

        if save_trace:
            trace_path = TRACES_DIR / f"{session_id}_{model_name}_{batch_size}_{precision}.csv"
            pd.DataFrame(power_samples).to_csv(trace_path, index=False)

        del model, x, out
        torch.cuda.empty_cache()

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        result.update({"valid": False, "warnings": "OOM"})
    except Exception as ex:
        torch.cuda.empty_cache()
        result.update({"valid": False, "warnings": f"exception:{type(ex).__name__}:{ex}"})
        traceback.print_exc()

    return result

print("Single-configuration measurement function defined.")

In [ ]:
#Cell 9 — Gate-C test (ResNet-18, batch 8, FP32)

print("Running Gate-C test: ResNet-18, batch_size=8, FP32, 60s window...")
gate_c_result = run_single_configuration(
    model_name="resnet18", batch_size=8, precision="fp32",
    measurement_seconds=60.0, session_id="gate_c"
)

print(json.dumps(gate_c_result, indent=2, default=str))

gate_c_pass = bool(gate_c_result["valid"]) and gate_c_result["num_power_samples"] >= MIN_POWER_SAMPLES_REQUIRED
print("\nGATE-C RESULT:", "PASS" if gate_c_pass else "FAIL")
if not gate_c_pass:
    print("Reason(s):", gate_c_result["warnings"])
else:
    print(f"Energy/sample: {gate_c_result['energy_per_sample_j']:.5f} J | "
          f"Throughput: {gate_c_result['throughput_samples_s']:.1f} samples/s | "
          f"Avg power: {gate_c_result['average_power_w']:.1f} W")
print("\nOnly proceed to Cell 10 (full experiment) if Gate-C == PASS.")

In [ ]:
#cell-10

def load_existing_results():
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame(columns=RESULT_FIELDS)

def config_already_done(existing_df, session_id, model, batch_size, precision):
    if existing_df.empty:
        return False
    mask = (
        (existing_df["session_id"] == session_id) &
        (existing_df["model"] == model) &
        (existing_df["batch_size"] == batch_size) &
        (existing_df["precision"] == precision)
    )
    return mask.any()

def append_result_to_csv(result):
    file_exists = RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow({k: result.get(k, "") for k in RESULT_FIELDS})

def run_full_session(session_id=None, seed=None):
    session_id, seed, grid = new_session(session_id, seed)
    total = len(grid)
    existing_df = load_existing_results()

    print(f"Starting session {session_id} | seed={seed} | {total} configurations")
    for i, cfg in enumerate(grid, start=1):
        if config_already_done(existing_df, session_id, cfg["model"], cfg["batch_size"], cfg["precision"]):
            print(f"[{i}/{total}] SKIP (already complete): {cfg}")
            continue

        t0 = time.time()
        result = run_single_configuration(
            model_name=cfg["model"], batch_size=cfg["batch_size"], precision=cfg["precision"],
            session_id=session_id,
        )
        result["random_seed"] = seed
        append_result_to_csv(result)
        existing_df = load_existing_results()
        elapsed = time.time() - t0
        remaining_est_min = (total - i) * (elapsed / 60.0)

        status = "OK" if result["valid"] else f"INVALID ({result['warnings']})"
        print(f"[{i}/{total}] {cfg['model']} b={cfg['batch_size']} {cfg['precision']} "
              f"-> {status} | {elapsed:.1f}s | ~{remaining_est_min:.1f} min remaining")

    print(f"\nSession {session_id} complete. Results saved to {RESULTS_CSV}")
    return session_id

# ---- THE ACTUAL GATE THAT WAS MISSING ----
if RUN_ENERGY_SWEEP:
    SESSION_ID = run_full_session(session_id=SESSION_ID, seed=SEED)
else:
    print("RUN_ENERGY_SWEEP is False — skipping the energy sweep for this run. "
          "Existing results in the CSV (if any) are left untouched.")

In [ ]:
#cell-11 — Accuracy evaluation on Imagenette v2 / Imagewoof v2 (10-class datasets)

import torchvision.datasets as tvd
import json, urllib.request, os
from pathlib import Path

DATASET_NAME = "imagewoof"
ACCURACY_BATCH_SIZE = 32

# No .tgz extraction needed this time — Imagewoof dataset is already unpacked
IMAGEWOOF_ROOT = Path("/kaggle/input/datasets/notsota/imagewoof/imagewoof2")
VALIDATION_DATA_ROOT = str(IMAGEWOOF_ROOT / "val")

# sanity check — confirm the structure before running anything expensive
print("Top-level contents:", os.listdir(IMAGEWOOF_ROOT))
print("Val folder contents:", os.listdir(VALIDATION_DATA_ROOT))

# ---- Standard Keras/PyTorch synset-id -> global ImageNet class index mapping ----
IMAGENET_CLASS_INDEX_URL = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"

def load_wnid_to_global_index():
    with urllib.request.urlopen(IMAGENET_CLASS_INDEX_URL) as f:
        class_idx = json.load(f)
    return {v[0]: int(k) for k, v in class_idx.items()}   # e.g. {"n01440764": 0, ...}

WNID_TO_GLOBAL_INDEX = load_wnid_to_global_index()

def build_accuracy_loader(model_name, root=VALIDATION_DATA_ROOT, batch_size=ACCURACY_BATCH_SIZE):
    transform = get_preprocessing_transform(model_name)
    dataset = tvd.ImageFolder(root=root, transform=transform)

    # dataset.class_to_idx maps folder name (WNID) -> LOCAL index (0-9)
    # Remap LOCAL -> GLOBAL so labels match the model's real 1000-way output
    local_to_global = {
        local_idx: WNID_TO_GLOBAL_INDEX[wnid]
        for wnid, local_idx in dataset.class_to_idx.items()
    }
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    return loader, local_to_global

def evaluate_accuracy(model_name):
    loader, local_to_global = build_accuracy_loader(model_name)
    model_fp32 = load_model_to_device(model_name, "fp32")
    model_fp16 = load_model_to_device(model_name, "fp16")

    records = []
    with torch.no_grad():
        for images, local_labels in loader:
            images_fp32 = images.to(DEVICE, dtype=torch.float32)
            images_fp16 = images_fp32.half()

            out32 = model_fp32(images_fp32)
            out16 = model_fp16(images_fp16)

            pred32 = out32.argmax(dim=1).cpu().numpy()
            pred16 = out16.argmax(dim=1).cpu().numpy()

            global_labels = [local_to_global[int(l)] for l in local_labels.numpy()]

            for p32, p16, lab in zip(pred32, pred16, global_labels):
                records.append({
                    'model': model_name, 'dataset': DATASET_NAME, 'label': int(lab),
                    'pred_fp32': int(p32), 'pred_fp16': int(p16),
                    'correct_fp32': int(p32 == lab), 'correct_fp16': int(p16 == lab),
                    'agree_fp32_fp16': int(p32 == p16),
                })

    del model_fp32, model_fp16
    torch.cuda.empty_cache()
    return pd.DataFrame(records)

def run_all_accuracy_evaluations(models=MODELS):
    all_dfs = []
    for m in models:
        print(f"Evaluating {DATASET_NAME} accuracy for {m}...")
        df = evaluate_accuracy(m)
        out_path = RESULTS_DIR / f"accuracy_{DATASET_NAME}_{m}.csv"
        df.to_csv(out_path, index=False)
        print(f"  {m}: top1_fp32={df.correct_fp32.mean():.4f} top1_fp16={df.correct_fp16.mean():.4f} "
              f"agreement={df.agree_fp32_fp16.mean():.4f}")
        all_dfs.append(df)
    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(RESULTS_DIR / f"accuracy_{DATASET_NAME}_all_models.csv", index=False)
    return combined

if RUN_ACCURACY_EVAL:
    accuracy_df = run_all_accuracy_evaluations()
else:
    print("RUN_ACCURACY_EVAL is False — skipping accuracy evaluation for this run.")

In [ ]:
#cell-12

df = load_existing_results()
print("Total rows:", len(df))
print("Valid rows:", df["valid"].astype(str).eq("True").sum() if not df.empty else 0)

if not df.empty:
    print("\nValidity rate by model/precision:")
    print(df.groupby(["model", "precision"])["valid"].apply(lambda s: (s.astype(str) == "True").mean()))

    display_cols = ["session_id", "model", "batch_size", "precision", "energy_per_sample_j",
                     "throughput_samples_s", "average_power_w", "valid", "warnings"]
    print("\nLast 20 rows:")
    print(df[display_cols].tail(20).to_string())

In [ ]:
#cell-13

import matplotlib.pyplot as plt

def load_valid_results():
    df = load_existing_results()
    df = df[df["valid"].astype(str) == "True"].copy()
    for col in ["batch_size", "energy_per_sample_j", "mean_latency_s", "throughput_samples_s",
                "peak_memory_gb", "average_power_w"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def analysis_energy_vs_batch(df):
    df = df.copy()
    df["energy_per_batch_j"] = df["energy_per_sample_j"] * df["batch_size"]
    fits = []
    for (model, prec), group in df.groupby(["model", "precision"]):
        if len(group) < 2:
            continue
        coeffs = np.polyfit(group["batch_size"], group["energy_per_batch_j"], 1)
        marginal, fixed = coeffs[0], coeffs[1]
        fits.append({"model": model, "precision": prec, "fixed_component_j": fixed, "marginal_component_j_per_sample": marginal})
    return pd.DataFrame(fits)

def analysis_precision_comparison(df):
    rows = []
    for model in df["model"].unique():
        for bs in df["batch_size"].unique():
            f32 = df[(df.model == model) & (df.batch_size == bs) & (df.precision == "fp32")]
            f16 = df[(df.model == model) & (df.batch_size == bs) & (df.precision == "fp16")]
            if f32.empty or f16.empty:
                continue
            e32, e16 = f32["energy_per_sample_j"].mean(), f16["energy_per_sample_j"].mean()
            l32, l16 = f32["mean_latency_s"].mean(), f16["mean_latency_s"].mean()
            t32, t16 = f32["throughput_samples_s"].mean(), f16["throughput_samples_s"].mean()
            m32, m16 = f32["peak_memory_gb"].mean(), f16["peak_memory_gb"].mean()
            rows.append({
                "model": model, "batch_size": bs,
                "energy_ratio_fp16_over_fp32": e16 / e32 if e32 else np.nan,
                "energy_reduction_pct": (1 - e16 / e32) * 100 if e32 else np.nan,
                "latency_ratio": l16 / l32 if l32 else np.nan,
                "throughput_ratio": t16 / t32 if t32 else np.nan,
                "memory_ratio": m16 / m32 if m32 else np.nan,
            })
    return pd.DataFrame(rows)

def analysis_pareto_frontier(df):
    results = {}
    for model, group in df.groupby("model"):
        pts = group[["batch_size", "precision", "energy_per_sample_j", "mean_latency_s"]].copy()
        pts = pts.dropna()
        is_dominated = []
        arr = pts[["energy_per_sample_j", "mean_latency_s"]].values
        for i in range(len(arr)):
            dominated = False
            for j in range(len(arr)):
                if i == j:
                    continue
                if (arr[j] <= arr[i]).all() and (arr[j] < arr[i]).any():
                    dominated = True
                    break
            is_dominated.append(dominated)
        pts["dominated"] = is_dominated
        results[model] = pts[~pts["dominated"]].reset_index(drop=True)
    return results

def analysis_session_repeatability(df):
    return df.groupby(["model", "batch_size", "precision"])["energy_per_sample_j"].agg(
        ["mean", "std", "count"]
    ).reset_index()

def analysis_accuracy_summary():
    acc_path = RESULTS_DIR / "accuracy_all_models.csv"
    if not acc_path.exists():
        print("No accuracy_all_models.csv found — skipping accuracy summary.")
        return None
    acc_df = pd.read_csv(acc_path)
    summary = acc_df.groupby("model").agg(
        top1_fp32=("correct_fp32", "mean"),
        top1_fp16=("correct_fp16", "mean"),
        agreement_fp32_fp16=("agree_fp32_fp16", "mean"),
        n_images=("label", "count"),
    ).reset_index()
    summary["accuracy_drop_pct"] = (summary["top1_fp32"] - summary["top1_fp16"]) * 100
    return summary

def plot_energy_vs_batch(df, model):
    sub = df[df.model == model]
    plt.figure(figsize=(6, 4))
    for prec, g in sub.groupby("precision"):
        g = g.sort_values("batch_size")
        plt.plot(g["batch_size"], g["energy_per_sample_j"], marker="o", label=prec)
    plt.xscale("log", base=2)
    plt.xlabel("Batch size")
    plt.ylabel("Energy per sample (J)")
    plt.title(f"Energy per sample vs batch size — {model}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"plot_energy_{model}.png")
    plt.show()

# ---- run analysis ----
results_df = load_valid_results()
energy_fits = analysis_energy_vs_batch(results_df)
precision_cmp = analysis_precision_comparison(results_df)
frontier = analysis_pareto_frontier(results_df)
repeatability = analysis_session_repeatability(results_df)
accuracy_summary = analysis_accuracy_summary()

print("Energy vs batch size fits (per model/precision):")
print(energy_fits.to_string())
print("\nFP32 vs FP16 comparison (energy/latency/throughput/memory):")
print(precision_cmp.to_string())
print("\nSession repeatability (mean/std/count of energy_per_sample_j):")
print(repeatability.to_string())
if accuracy_summary is not None:
    print("\nAccuracy summary (FP32 vs FP16):")
    print(accuracy_summary.to_string())

for m in MODELS:
    if not results_df[results_df.model == m].empty:
        plot_energy_vs_batch(results_df, m)

In [ ]:
#cell-14 — Baseline comparison + prediction agreement

def analysis_baseline_comparison(df):
    avg = df.groupby(['model','batch_size','precision']).agg(
        energy_per_sample_j=('energy_per_sample_j','mean'),
        mean_latency_s=('mean_latency_s','mean'),
        throughput_samples_s=('throughput_samples_s','mean'),
        peak_memory_gb=('peak_memory_gb','mean'),
    ).reset_index()

    baseline = avg[(avg.batch_size==1) & (avg.precision=='fp32')][
        ['model','energy_per_sample_j','mean_latency_s','throughput_samples_s','peak_memory_gb']
    ].rename(columns={
        'energy_per_sample_j':'baseline_energy', 'mean_latency_s':'baseline_latency',
        'throughput_samples_s':'baseline_throughput', 'peak_memory_gb':'baseline_memory',
    })

    merged = avg.merge(baseline, on='model')
    merged['energy_reduction_vs_baseline_pct'] = (1 - merged['energy_per_sample_j']/merged['baseline_energy'])*100
    merged['latency_reduction_vs_baseline_pct'] = (1 - merged['mean_latency_s']/merged['baseline_latency'])*100
    merged['throughput_gain_vs_baseline_x'] = merged['throughput_samples_s']/merged['baseline_throughput']
    merged['memory_change_vs_baseline_x'] = merged['peak_memory_gb']/merged['baseline_memory']
    return merged

def analysis_prediction_agreement():
    acc_path = RESULTS_DIR / "accuracy_all_models.csv"
    if not acc_path.exists():
        print("No accuracy_all_models.csv found — skipping.")
        return None
    acc = pd.read_csv(acc_path)
    rows = []
    for model, g in acc.groupby('model'):
        n = len(g)
        both_correct = ((g.correct_fp32==1) & (g.correct_fp16==1)).sum()
        both_wrong   = ((g.correct_fp32==0) & (g.correct_fp16==0)).sum()
        fp32_only    = ((g.correct_fp32==1) & (g.correct_fp16==0)).sum()
        fp16_only    = ((g.correct_fp32==0) & (g.correct_fp16==1)).sum()
        disagreements = (g.pred_fp32 != g.pred_fp16).sum()
        rows.append({
            'model': model, 'n_images': n,
            'agreement_pct': g.agree_fp32_fp16.mean()*100,
            'disagreements': disagreements,
            'both_correct_pct': both_correct/n*100,
            'both_wrong_pct': both_wrong/n*100,
            'fp32_only_correct_pct': fp32_only/n*100,
            'fp16_only_correct_pct': fp16_only/n*100,
        })
    return pd.DataFrame(rows)

# ---- run ----
baseline_comparison = analysis_baseline_comparison(results_df)
prediction_agreement = analysis_prediction_agreement()

print("Baseline comparison (batch=1, FP32 as baseline):")
print(baseline_comparison.to_string())

baseline_comparison.to_csv(RESULTS_DIR / "baseline_comparison.csv", index=False)

if prediction_agreement is not None:
    print("\nPrediction agreement (FP32 vs FP16):")
    print(prediction_agreement.to_string(index=False))
    prediction_agreement.to_csv(RESULTS_DIR / "prediction_agreement.csv", index=False)

In [ ]:
#cell-15 — Pareto frontier export

def analysis_pareto_frontier_export(results_df):
    avg = results_df.groupby(['model','batch_size','precision']).agg(
        energy_per_sample_j=('energy_per_sample_j','mean'),
        mean_latency_s=('mean_latency_s','mean'),
    ).reset_index()

    def pareto_frontier(group):
        pts = group.reset_index(drop=True)
        arr = pts[['energy_per_sample_j','mean_latency_s']].values
        dominated = []
        for i in range(len(arr)):
            is_dom = False
            for j in range(len(arr)):
                if i == j:
                    continue
                if (arr[j] <= arr[i]).all() and (arr[j] < arr[i]).any():
                    is_dom = True
                    break
            dominated.append(is_dom)
        pts['dominated'] = dominated
        return pts[~pts['dominated']].drop(columns='dominated')

    all_frontiers = []
    for model, g in avg.groupby('model'):
        fr = pareto_frontier(g)
        fr['model'] = model
        all_frontiers.append(fr)
        print(f"\n=== Pareto-optimal configs for {model} ===")
        print(fr[['batch_size','precision','energy_per_sample_j','mean_latency_s']]
              .sort_values('batch_size').to_string(index=False))

    return pd.concat(all_frontiers, ignore_index=True)

pareto_frontier_df = analysis_pareto_frontier_export(results_df)
pareto_frontier_df.to_csv(RESULTS_DIR / "pareto_frontier.csv", index=False)
print("\nSaved: results/pareto_frontier.csv")

In [ ]:
#cell-16 — McNemar's exact test (paired FP32 vs FP16 correctness)

from scipy.stats import binomtest

def analysis_mcnemar_test():
    acc_path = RESULTS_DIR / "accuracy_all_models.csv"
    if not acc_path.exists():
        print("No accuracy_all_models.csv found — skipping McNemar's test.")
        return None

    acc = pd.read_csv(acc_path)
    results = []
    for model, g in acc.groupby('model'):
        both_correct = ((g.correct_fp32 == 1) & (g.correct_fp16 == 1)).sum()
        fp32_only    = ((g.correct_fp32 == 1) & (g.correct_fp16 == 0)).sum()
        fp16_only    = ((g.correct_fp32 == 0) & (g.correct_fp16 == 1)).sum()
        both_wrong   = ((g.correct_fp32 == 0) & (g.correct_fp16 == 0)).sum()

        n_discordant = fp32_only + fp16_only
        if n_discordant > 0:
            p_value = binomtest(min(fp32_only, fp16_only), n_discordant, 0.5,
                                 alternative='two-sided').pvalue
        else:
            p_value = 1.0  # no discordant pairs at all -> no evidence of a difference

        results.append({
            'model': model, 'n': len(g),
            'both_correct': both_correct, 'both_wrong': both_wrong,
            'fp32_only_correct': fp32_only, 'fp16_only_correct': fp16_only,
            'n_discordant': n_discordant,
            'mcnemar_exact_p_value': p_value,
            'significant_at_0.05': p_value < 0.05,
        })
    return pd.DataFrame(results)

mcnemar_df = analysis_mcnemar_test()
if mcnemar_df is not None:
    print("McNemar's exact test — FP32 vs FP16 paired correctness:")
    print(mcnemar_df.to_string(index=False))
    mcnemar_df.to_csv(RESULTS_DIR / "mcnemar_test.csv", index=False) 

In [ ]:
#cell-17 — Additional summary plots for the paper

import matplotlib.pyplot as plt
import numpy as np

# ---- Plot 1: FP16 energy reduction % by model (bar chart) ----
piv = results_df.groupby(['model','batch_size','precision'])['energy_per_sample_j'].mean().reset_index()
piv2 = piv.pivot_table(index=['model','batch_size'], columns='precision', values='energy_per_sample_j')
piv2['reduction_pct'] = (1 - piv2['fp16']/piv2['fp32']) * 100
avg_reduction = piv2.groupby('model')['reduction_pct'].mean().sort_values()

plt.figure(figsize=(8,5))
plt.barh(avg_reduction.index, avg_reduction.values, color='#2a9d8f')
plt.xlabel('Mean FP16 energy reduction vs FP32 (%)')
plt.title('FP16 Energy Reduction by Model (averaged across batch sizes)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_fp16_reduction_by_model.png", dpi=150)
plt.show()

# ---- Plot 2: Energy vs Latency scatter with Pareto frontier ----
fig, axes = plt.subplots(2, 3, figsize=(16,9))
axes = axes.flatten()
for i, model in enumerate(MODELS):
    ax = axes[i]
    sub = piv.merge(results_df.groupby(['model','batch_size','precision'])['mean_latency_s'].mean().reset_index(),
                     on=['model','batch_size','precision'])
    sub = sub[sub.model == model]
    for prec, color in [('fp32','#e76f51'), ('fp16','#2a9d8f')]:
        s = sub[sub.precision == prec]
        ax.scatter(s['mean_latency_s'], s['energy_per_sample_j'], label=prec, color=color, s=40)
    # highlight pareto points for this model
    if model in pareto_frontier_df['model'].values:
        pf = pareto_frontier_df[pareto_frontier_df.model == model].sort_values('mean_latency_s')
        ax.plot(pf['mean_latency_s'], pf['energy_per_sample_j'], 'k--', linewidth=1, label='Pareto frontier')
    ax.set_title(model)
    ax.set_xlabel('Latency (s)')
    ax.set_ylabel('Energy/sample (J)')
    ax.legend(fontsize=8)
plt.suptitle('Energy vs Latency Trade-off with Pareto Frontier')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_pareto_scatter.png", dpi=150)
plt.show()

# ---- Plot 3: Best energy reduction vs baseline, per model ----
best = baseline_comparison.loc[baseline_comparison.groupby('model')['energy_reduction_vs_baseline_pct'].idxmax()]
best = best.sort_values('energy_reduction_vs_baseline_pct')

fig, ax1 = plt.subplots(figsize=(9,5))
ax2 = ax1.twinx()
x = np.arange(len(best))
ax1.bar(x - 0.2, best['energy_reduction_vs_baseline_pct'], width=0.4, color='#264653', label='Energy reduction (%)')
ax2.bar(x + 0.2, best['throughput_gain_vs_baseline_x'], width=0.4, color='#e9c46a', label='Throughput gain (x)')
ax1.set_xticks(x)
ax1.set_xticklabels(best['model'], rotation=30, ha='right')
ax1.set_ylabel('Energy reduction vs baseline (%)')
ax2.set_ylabel('Throughput gain (x)')
plt.title('Best Configuration vs Batch=1/FP32 Baseline')
fig.legend(loc='upper left', bbox_to_anchor=(0.1,0.9))
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_baseline_comparison.png", dpi=150)
plt.show()

# ---- Plot 4: Repeatability across sessions (error bars) ----
rep_detail = results_df[results_df.precision=='fp16'].groupby(['model','batch_size'])['energy_per_sample_j'].agg(['mean','std']).reset_index()
rep_bs8 = rep_detail[rep_detail.batch_size==8].sort_values('model')

plt.figure(figsize=(8,5))
plt.bar(rep_bs8['model'], rep_bs8['mean'], yerr=rep_bs8['std'], capsize=5, color='#457b9d')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Energy per sample (J)')
plt.title('Session-to-Session Variation at Batch=8, FP16 (error bars = std across sessions)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / "plot_repeatability.png", dpi=150)
plt.show()

# ---- Plot 5: FP32 vs FP16 accuracy per model ----
acc_path = RESULTS_DIR / "accuracy_all_models.csv"
if acc_path.exists():
    acc = pd.read_csv(acc_path)
    acc_summary = acc.groupby('model').agg(top1_fp32=('correct_fp32','mean'), top1_fp16=('correct_fp16','mean')).reset_index()
    x = np.arange(len(acc_summary))
    plt.figure(figsize=(9,5))
    plt.bar(x - 0.2, acc_summary['top1_fp32'], width=0.4, label='FP32', color='#e76f51')
    plt.bar(x + 0.2, acc_summary['top1_fp16'], width=0.4, label='FP16', color='#2a9d8f')
    plt.xticks(x, acc_summary['model'], rotation=30, ha='right')
    plt.ylabel('Top-1 Accuracy')
    plt.title('FP32 vs FP16 Accuracy by Model (Imagenette + Imagewoof)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "plot_accuracy_comparison.png", dpi=150)
    plt.show()
else:
    print("accuracy_imagenette_imagewoof_combined.csv not found — skipping accuracy plot.")

print("\nAll 5 additional plots saved to results/ folder.")

In [ ]:
#cell-18 — Formal repeatability significance test (Friedman + pairwise Wilcoxon)

from scipy.stats import friedmanchisquare, wilcoxon
from itertools import combinations

def analysis_repeatability_significance(results_df):
    piv = results_df.pivot_table(index=['model','batch_size','precision'],
                                   columns='session_id', values='energy_per_sample_j')
    piv = piv.dropna()
    sessions = sorted(piv.columns)

    stat, p = friedmanchisquare(*[piv[s] for s in sessions])
    friedman_result = pd.DataFrame([{
        'test': 'Friedman', 'statistic': stat, 'p_value': p,
        'n_configs': len(piv), 'n_sessions': len(sessions)
    }])

    pair_results = []
    for s1, s2 in combinations(sessions, 2):
        stat_w, p_w = wilcoxon(piv[s1], piv[s2])
        pair_results.append({'session_1': s1, 'session_2': s2,
                              'wilcoxon_stat': stat_w, 'p_value': round(p_w, 4),
                              'significant': p_w < 0.05})
    pairwise_df = pd.DataFrame(pair_results)
    return friedman_result, pairwise_df

friedman_result, pairwise_wilcoxon = analysis_repeatability_significance(results_df)
print("=== FRIEDMAN TEST (overall session effect) ===")
print(friedman_result.to_string(index=False))
print("\n=== PAIRWISE WILCOXON SIGNED-RANK TESTS ===")
print(pairwise_wilcoxon.to_string(index=False))

friedman_result.to_csv(RESULTS_DIR / "repeatability_friedman_test.csv", index=False)
pairwise_wilcoxon.to_csv(RESULTS_DIR / "repeatability_wilcoxon_pairwise.csv", index=False)

In [ ]:
#cell-19 — Cross-validation against CodeCarbon (independent energy tool)

!pip install codecarbon --break-system-packages -q

from codecarbon import EmissionsTracker
import time

def run_codecarbon_comparison(model_name="resnet18", batch_size=8, precision="fp32",
                                duration_seconds=60.0, n_repeats=3):
    """Runs the same workload multiple times, once measured by our own method,
    once measured by CodeCarbon, for direct comparison."""
    comparison_results = []

    for repeat in range(n_repeats):
        print(f"\n--- Repeat {repeat+1}/{n_repeats} ---")

        # ---- Our own method (reuses existing run_single_configuration from Cell 8) ----
        our_result = run_single_configuration(
            model_name=model_name, batch_size=batch_size, precision=precision,
            measurement_seconds=duration_seconds, session_id=f"codecarbon_check_{repeat}"
        )
        our_energy_j = our_result['energy_j']

        # ---- CodeCarbon's independent measurement, same workload ----
        model = load_model_to_device(model_name, precision)
        dtype = torch.float16 if precision == "fp16" else torch.float32
        x = torch.randn(batch_size, 3, 224, 224, device=DEVICE, dtype=dtype)
        with torch.no_grad():
            for _ in range(30):
                model(x)
        torch.cuda.synchronize(DEVICE)

        tracker = EmissionsTracker(measure_power_secs=1, log_level="error", save_to_file=False)
        tracker.start()
        start = time.time()
        with torch.no_grad():
            while (time.time() - start) < duration_seconds:
                model(x)
                torch.cuda.synchronize(DEVICE)
        tracker.stop()
        codecarbon_energy_j = tracker._total_energy.kWh * 3_600_000  # kWh -> Joules

        pct_diff = abs(our_energy_j - codecarbon_energy_j) / our_energy_j * 100
        comparison_results.append({
            'repeat': repeat + 1, 'model': model_name, 'batch_size': batch_size, 'precision': precision,
            'our_energy_j': our_energy_j, 'codecarbon_energy_j': codecarbon_energy_j,
            'pct_difference': pct_diff
        })
        print(f"Our method: {our_energy_j:.2f} J | CodeCarbon: {codecarbon_energy_j:.2f} J | Diff: {pct_diff:.1f}%")

        del model, x
        torch.cuda.empty_cache()

    return pd.DataFrame(comparison_results)

if RUN_CODECARBON_VALIDATION:
    validation_df = run_codecarbon_comparison()
    print("\n=== SUMMARY ===")
    print(validation_df.to_string(index=False))
    print(f"\nMean % difference: {validation_df['pct_difference'].mean():.1f}%")
    validation_df.to_csv(RESULTS_DIR / "codecarbon_validation.csv", index=False)
else:
    print("RUN_CODECARBON_VALIDATION is False — skipping CodeCarbon comparison for this run.")

In [ ]:
#cell-20 — Scope-matched CodeCarbon validation (GPU-only comparison)

from codecarbon import EmissionsTracker
import time

def run_codecarbon_gpu_only_comparison(model_name="resnet18", batch_size=8, precision="fp32",
                                         duration_seconds=60.0, n_repeats=3):
    comparison_results = []

    for repeat in range(n_repeats):
        print(f"\n--- Repeat {repeat+1}/{n_repeats} ---")

        # ---- Our own method ----
        our_result = run_single_configuration(
            model_name=model_name, batch_size=batch_size, precision=precision,
            measurement_seconds=duration_seconds, session_id=f"codecarbon_gpuonly_check_{repeat}"
        )
        our_energy_j = our_result['energy_j']

        # ---- CodeCarbon, restricted to GPU index 0 only ----
        model = load_model_to_device(model_name, precision)
        dtype = torch.float16 if precision == "fp16" else torch.float32
        x = torch.randn(batch_size, 3, 224, 224, device=DEVICE, dtype=dtype)
        with torch.no_grad():
            for _ in range(30):
                model(x)
        torch.cuda.synchronize(DEVICE)

        tracker = EmissionsTracker(
            measure_power_secs=1,
            log_level="error",
            save_to_file=False,
            gpu_ids=[0],          # restrict to GPU 0 only, matching our own methodology
            tracking_mode="process"
        )
        tracker.start()
        start = time.time()
        with torch.no_grad():
            while (time.time() - start) < duration_seconds:
                model(x)
                torch.cuda.synchronize(DEVICE)
        tracker.stop()

        # Pull the GPU-ONLY component, not the combined CPU+GPU+RAM total
        emissions_data = tracker.final_emissions_data
        codecarbon_gpu_energy_j = emissions_data.gpu_energy * 3_600_000       # kWh -> J
        codecarbon_total_energy_j = emissions_data.energy_consumed * 3_600_000  # for reference

        pct_diff_gpu_only = abs(our_energy_j - codecarbon_gpu_energy_j) / our_energy_j * 100

        comparison_results.append({
            'repeat': repeat + 1, 'model': model_name, 'batch_size': batch_size, 'precision': precision,
            'our_energy_j': our_energy_j,
            'codecarbon_gpu_only_energy_j': codecarbon_gpu_energy_j,
            'codecarbon_total_energy_j': codecarbon_total_energy_j,
            'pct_difference_gpu_only': pct_diff_gpu_only
        })
        print(f"Our method (GPU-only): {our_energy_j:.2f} J | "
              f"CodeCarbon GPU-only: {codecarbon_gpu_energy_j:.2f} J | "
              f"CodeCarbon total (CPU+GPU+RAM): {codecarbon_total_energy_j:.2f} J | "
              f"GPU-only diff: {pct_diff_gpu_only:.1f}%")

        del model, x
        torch.cuda.empty_cache()

    return pd.DataFrame(comparison_results)
    
if RUN_CODECARBON_VALIDATION:
    validation_gpu_only_df = run_codecarbon_gpu_only_comparison()
    print("\n=== SUMMARY (scope-matched, GPU-only) ===")
    print(validation_gpu_only_df.to_string(index=False))
    print(f"\nMean GPU-only % difference: {validation_gpu_only_df['pct_difference_gpu_only'].mean():.2f}%")
    validation_gpu_only_df.to_csv(RESULTS_DIR / "codecarbon_validation_gpu_only.csv", index=False)
else:
    print("RUN_CODECARBON_VALIDATION is False — skipping GPU-only CodeCarbon comparison for this run.")

In [ ]:
#cell-21 — Factorial ANOVA on log(energy per sample) with interactions

import numpy as np
from scipy.stats import f as f_dist

def run_factorial_anova(results_df):
    df = results_df[results_df['valid']==True].copy()
    df['log_energy'] = np.log(df['energy_per_sample_j'])
    y = df['log_energy'].values
    n = len(df)

    def dm(cols):
        X = np.ones((len(df), 1))
        for c in cols:
            X = np.hstack([X, pd.get_dummies(df[c], drop_first=True, prefix=c).astype(float).values])
        return X

    def inter(groups):
        parts = []
        for group in groups:
            dfs = [pd.get_dummies(df[c], drop_first=True, prefix=c).astype(float) for c in group]
            m = dfs[0]
            for d in dfs[1:]:
                m = pd.DataFrame({f"{a}*{b}": m[a].values*d[b].values for a in m.columns for b in d.columns})
            parts.append(m.values)
        return np.hstack(parts) if parts else np.zeros((len(df),0))

    def rss(X):
        coef, *_ = np.linalg.lstsq(X, y, rcond=None)
        return np.sum((y - X@coef)**2), np.linalg.matrix_rank(X)

    X0 = dm([])
    X1 = dm(['model'])
    X2 = dm(['model','batch_size'])
    X3 = dm(['model','batch_size','precision'])
    X4 = np.hstack([X3, inter([('model','precision')])])
    X5 = np.hstack([X4, inter([('model','batch_size')])])
    X6 = np.hstack([X5, inter([('batch_size','precision')])])
    X7 = np.hstack([X6, inter([('model','batch_size','precision')])])

    RSS0,_ = rss(X0); RSS1,_ = rss(X1); RSS2,_ = rss(X2); RSS3,_ = rss(X3)
    RSS4,_ = rss(X4); RSS5,_ = rss(X5); RSS6,_ = rss(X6); RSS7, r7 = rss(X7)

    df_error = n - r7
    MSE = RSS7 / df_error

    terms = [
        ('Model', RSS0-RSS1, 5), ('Batch size', RSS1-RSS2, 7), ('Precision', RSS2-RSS3, 1),
        ('Model x Precision', RSS3-RSS4, 5), ('Model x Batch', RSS4-RSS5, 35),
        ('Batch x Precision', RSS5-RSS6, 7), ('Model x Batch x Precision', RSS6-RSS7, 35),
    ]
    rows = []
    for name, SS, dfree in terms:
        MS = SS/dfree; F = MS/MSE; eta2 = SS/(SS+RSS7)
        p = 1 - f_dist.cdf(F, dfree, df_error)
        rows.append({'Factor':name,'df':dfree,'F':round(F,2),'p_value':p,'partial_eta2':round(eta2,4)})
    return pd.DataFrame(rows)

anova_results = run_factorial_anova(results_df)
print(anova_results.to_string(index=False))
anova_results.to_csv(RESULTS_DIR / "factorial_anova.csv", index=False)